In [1]:
from pathlib import Path
import pandas as pd
from openai import OpenAI
import re
# from anthropic import Anthropic
    
# 60 questions

# read the Excel file, first sheet
df = pd.read_excel("Fine-tuning instruction set_corrected.xlsx", sheet_name=0)

print(df.columns)

# Rename columns for clarity
human_df_renamed = df.rename(columns={
    'Evidence': 'Human_Evidence',
    'Rationale': 'Human_Rationale',
    'Answer': 'Human_Answer'
})

print(human_df_renamed.columns)


Index(['PMID', 'QID', 'Question', 'Evidence', 'Rationale', 'Answer', 'Done',
       'Comment'],
      dtype='object')
Index(['PMID', 'QID', 'Question', 'Human_Evidence', 'Human_Rationale',
       'Human_Answer', 'Done', 'Comment'],
      dtype='object')


In [10]:



# --- Collect all merged data ---
merged_list = []

model = "gpt-5-3"
# model = "claude-opus-4-1-20250805-1"
base_dir = Path(f"Results_19/{model}")

for tmp_folder in base_dir.iterdir():
    if not tmp_folder.is_dir() or tmp_folder.name == ".DS_Store":
        continue

    file_path = tmp_folder / "gpt_answers_19.xlsx"
    if not file_path.exists():
        continue

    model_df = pd.read_excel(file_path)

    # ✅ Replace wrong PMID with folder name (convert to int safely)
    try:
        model_df["PMID"] = int(tmp_folder.name)
    except ValueError:
        # skip if folder name isn't a valid number
        continue

    model_df_renamed = model_df.rename(columns={
        'Evidence': f'{model}_Evidence',
        'Rationale': f'{model}_Rationale',
        'Answer': f'{model}_Answer'
    })

    print(f"Processing folder: {tmp_folder.name}")
    print("Human columns:", human_df_renamed.columns.tolist())
    print("Model columns:", model_df_renamed.columns.tolist())
    merged_df = pd.merge(
        human_df_renamed,
        model_df_renamed,
        on=['PMID', 'QID'],
        how='inner'
    )
    # Combine Question_x and Question_y — take whichever is not null
    merged_df["Question"] = merged_df["Question_x"].combine_first(merged_df["Question_y"])

    merged_list.append(merged_df)

with open("questions.txt", "r", encoding="utf-8") as f:
    questions_block = f.read()
questions_block = questions_block.split("\n")

# Combine all merged dataframes vertically (stack)
if merged_list:
    final_df = pd.concat(merged_list, ignore_index=True)
    final_df = final_df[['PMID', 'QID', 'Question',
                         'Human_Evidence', 'Human_Rationale', 'Human_Answer',
                         f'{model}_Evidence', f'{model}_Rationale', f'{model}_Answer']]

    output_path = Path(f"Results_19/{model}/merged_all.xlsx")
    print(final_df.columns)
    ## replace questions

    
    # # 60 questions

    # # read the Excel file, first sheet
    # df = pd.read_excel("Fine-tuning instruction set, Aug 22.xlsx", sheet_name=0)

    # # filter rows where PMID = 19686436
    # filtered_df = df[df["PMID"] == int(19686436)]

    # combined_list = filtered_df.apply(lambda row: f"{row['QID']}: {row['Question']}", axis=1).tolist()

    # # Join into one string with newline separators
    # questions_block = "\n".join(combined_list)

    # # print(questions_block)

    # Create a mapping from QID → Question in the filtered df
    
    qid_to_question = {}
    for item in questions_block:
        # Split on the first ": " to get QID and Question
        qid, question = item.split(": ", 1)
        qid_to_question[qid] = question

    # Replace questions in final_df based on QID
    # final_df["Question"] = final_df["QID"].map(qid_to_question).fillna(final_df["Question"])
    final_df["Question"] = final_df["QID"].astype(str).map(qid_to_question).fillna(final_df["Question"])



    final_df.to_excel(output_path, index=False)
    print(f"✅ Vertically stacked Excel saved to: {output_path}")
else:
    print("⚠️ No gpt_answers_19.xlsx files found.")


Processing folder: 20643915
Human columns: ['PMID', 'QID', 'Question', 'Human_Evidence', 'Human_Rationale', 'Human_Answer', 'Done', 'Comment']
Model columns: ['PMID', 'QID', 'Question', 'gpt-5-3_Evidence', 'gpt-5-3_Rationale', 'gpt-5-3_Answer']
Processing folder: 25560398
Human columns: ['PMID', 'QID', 'Question', 'Human_Evidence', 'Human_Rationale', 'Human_Answer', 'Done', 'Comment']
Model columns: ['PMID', 'QID', 'Question', 'gpt-5-3_Evidence', 'gpt-5-3_Rationale', 'gpt-5-3_Answer']
Processing folder: 25653132
Human columns: ['PMID', 'QID', 'Question', 'Human_Evidence', 'Human_Rationale', 'Human_Answer', 'Done', 'Comment']
Model columns: ['PMID', 'QID', 'Question', 'gpt-5-3_Evidence', 'gpt-5-3_Rationale', 'gpt-5-3_Answer']
Processing folder: 20483657
Human columns: ['PMID', 'QID', 'Question', 'Human_Evidence', 'Human_Rationale', 'Human_Answer', 'Done', 'Comment']
Model columns: ['PMID', 'QID', 'Question', 'gpt-5-3_Evidence', 'gpt-5-3_Rationale', 'gpt-5-3_Answer']
Processing folder: 2

Manual Evaluation - todo

In [70]:
import pandas as pd
import unicodedata
import re
# Load Excel
# Don't truncate strings in cells
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)

df = pd.read_excel("Results_19/gpt-5-3/merged_all.xlsx")

# QIDs that are boolean
boolean_qids = [1,2,9,10,12,13,14,16,19]

def normalize_text(s):
    if pd.isna(s):
        return ""
    # Convert everything to string first
    s = str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKC", s)

    # Remove commas from numbers
    s = s.replace(",", "")

    # Normalize date ranges and dashes
    s = re.sub(r"\bto\b", "-", s)
    s = re.sub(r"[–—−]", "-", s)
    s = re.sub(r"\s*-\s*", "-", s)

    # Normalize separators
    s = re.sub(r"[;,/]", " ", s)
    s = re.sub(r"\s+", " ", s)

    # Domain-specific substitutions
    s = re.sub(r"\breverse transcriptase\b", "rt", s)
    s = re.sub(r"\bprotease\b", "pr", s)
    s = re.sub(r"\bintegrase\b", "in", s)
    s = re.sub(r"\bnext[- ]generation sequencing\b", "ngs", s)

    # Drug classes (singular form)
    drug_classes = ["nrtis", "nnrtis", "pis", "instis", "entry inhibitors", "fusion inhibitors"]
    for cls in drug_classes:
        singular = cls.rstrip("s")
        s = re.sub(rf"\b{cls}\b", singular, s)

    return s.strip()


# Function to compare boolean answers
def compare_boolean(human, gpt):
    # Handle NaN
    if pd.isna(human) and pd.isna(gpt):
        return True
    if pd.isna(human) and str(gpt).strip().lower() == "not reported":
        return True
    if pd.isna(gpt) and str(human).strip().lower() == "not reported":
        return True
    # Normalize values
    human_norm = str(human).strip().lower()
    gpt_norm = str(gpt).strip().lower()
    
    # if human_norm not in ["yes", "no", "none", "not reported"] or gpt_norm not in ["yes", "no", "not reported"]:
    #     return "invalid"
    return human_norm == gpt_norm

# Function to compare categorical answers (exact match)
def compare_categorical(human, gpt):
    human_norm = normalize_text(human)
    gpt_norm = normalize_text(gpt)
    
    # --- Domain expansion for HIV genes ---
    # Treat 'pol' as equivalent to containing 'pr' and 'rt'
    if ("pol" in human_norm and ("pr" in gpt_norm or "rt" in gpt_norm)) or \
       ("pol" in gpt_norm and ("pr" in human_norm or "rt" in human_norm)):
        return True
    
    # --- Illumina ↔ NGS equivalence ---
    if ("illumina" in human_norm and "ngs" in gpt_norm) or \
       ("illumina" in gpt_norm and "ngs" in human_norm):
        return True
    
    if ("full length" in human_norm and "gag" in gpt_norm and "pol" in gpt_norm and "env" in gpt_norm) or \
       ("full length" in gpt_norm and "gag" in human_norm and "pol" in human_norm and "env" in human_norm):
        return True
    
    # Exact match
    if str(human_norm) == str(gpt_norm):
        return True
    
    # One string includes the other
    if human_norm in gpt_norm or gpt_norm in human_norm:
        return True
    
    if pd.isna(human_norm) and gpt_norm == "not reported":
        return True
    if pd.isna(gpt_norm) and human_norm == "not reported":
        return True
    
    return False




# Define edge-case mappings
EDGE_CASES = {
    "hong kong sar": "hong kong sar china",
}

def normalize_list(s):
    if pd.isna(s):
        return set()
    s = str(s).strip()
    # Remove parentheses and content inside
    s = re.sub(r"\([^)]*\)", "", s)
    # Split by commas or semicolons
    items = re.split(r"[;,]", s)
    # Lowercase and strip spaces
    items = [item.strip().lower() for item in items if item.strip()]
    # Convert to set (order-independent)
    # Apply edge-case mapping
    items = [EDGE_CASES.get(item, item) for item in items]
    return set(items)

def compare_lists(human, gpt):
    print(normalize_list(human))
    print(normalize_list(gpt))
    return normalize_list(human) == normalize_list(gpt)

# human = "Hong Kong SAR (China), Thailand (Bangkok and Chiang Mai), Philippines"
# gpt   = "Hong Kong SAR China; Thailand; Philippines"
human = "2,439"
gpt = 2439
print(compare_categorical(human, gpt))
# compare_lists(human, gpt)


def compare_lists(human, gpt):
    human_set = normalize_list(human)
    gpt_set = normalize_list(gpt)
    if pd.isna(human) and str(gpt).strip().lower() == "not reported":
        return True
    if pd.isna(gpt) and str(human).strip().lower() == "not reported":
        return True
    return human_set == gpt_set

def expand_accessions(accession_str):
    """Convert accession strings/ranges to a set of accession IDs."""
    if accession_str is None:
        return set()
    if (isinstance(accession_str, float) and pd.isna(accession_str)) \
       or (isinstance(accession_str, str) and accession_str.strip().lower() in ["not reported", "none"]):
        return set()
    
    accession_str = str(accession_str)
    accession_str = accession_str.replace("–", "-").replace(" to ", "-")  # normalize
    accession_str = accession_str.replace(" ", "")  # remove spaces
    accession_str = re.sub(r"-+", "-", accession_str)  # replace one or more dashes with a single dash
    # Remove any content in parentheses
    accession_str = re.sub(r"\(.*?\)", "", accession_str)
    accession_str = re.sub(r"(?i)\band\b", ";", accession_str) # convert and to ;
    accession_str = re.sub(r"(?i)and", ";", accession_str)
    items = re.split(r"[;,]", accession_str)  # split by ; or ,
    
    accessions = set()
    
    for item in items:
        if '-' in item:
            # print(item, accession_str)
            start, end = item.split('-')
            # Extract prefix and number
            prefix_start, num_start = re.match(r"([A-Z]+)(\d+)", start).groups()
            prefix_end, num_end = re.match(r"([A-Z]+)(\d+)", end).groups()
            
            if prefix_start != prefix_end:
                # Different prefixes, treat them separately
                accessions.add(start)
                accessions.add(end)
            else:
                width = len(num_start)  # preserve leading zeros
                for n in range(int(num_start), int(num_end)+1):
                    accessions.add(f"{prefix_start}{str(n).zfill(width)}")
        else:
            accessions.add(item)
    
    return accessions


def compare_accessions(acc1, acc2):
    """Compare two accession representations. Returns True if they represent the same set."""
    set1 = expand_accessions(acc1)
    set2 = expand_accessions(acc2)
    # print(set1, set2)
    return set1 == set2

# acc1 = "KJ820145-KJ820408, KP234525-KP234916, KP234917-KP235200"
# acc2 = "KJ820145–KJ820408; KP234525–KP234916; KP234917–KP235200"

# print(compare_accessions(acc1, acc2))

# Apply comparison
def evaluate_row(row):
    human_row = row['Human_Answer']
    ai_row = row['gpt-5-3_Answer']
    if row['QID'] in boolean_qids:
        return compare_boolean(human_row, ai_row)
    if str(row['QID']) in ["18", "6"]:
        return compare_lists(human_row, ai_row)
    if str(row["QID"]) == "3":
        return compare_accessions(human_row, ai_row)
    else:
        return compare_categorical(human_row, ai_row)

df['Evaluation'] = df.apply(evaluate_row, axis=1)

# Optional: calculate summary
summary = df['Evaluation'].value_counts()
print(summary)
false_df = df.loc[df["Evaluation"] == False, ["QID", "Human_Answer", "gpt-5-3_Answer", "Evaluation"]]



# # Save results
false_df.to_excel("evaluation_results.xlsx", index=False)


True
True     835
False    172
Name: Evaluation, dtype: int64


In [ ]:
from openai import OpenAI

model = "deepseek-chat"

client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com"
)

def auto_eval_answer(folder_name = "gpt-5", model='gpt-5'):

    records = pd.read_excel(f"Results_19/{folder_name}/merged_all.xlsx")
    count = 0
    for idx, row in records.iterrows():
        q = row['Question']
        evidence1 = row["Human_Evidence"]
        rationale1 = row["Human_Rationale"]
        a1 = row['Human_Answer']
        evidence2 = row[f"{folder_name}_Evidence"]
        rationale2 = row[f"{folder_name}_Rationale"]
        a2 = row[f'{folder_name}_Answer']

        prompt = f"""
You're an expert in HBV drug resistance research.

I have one question (Q) and two answers:
- A1 (answer key)
- A2 

Help me evaluate if A2 is correct based on A1 and the following rules. Only reply "Yes" (correct) or "No" (incorrect).

Rules:

1. If A2 includes or covers the content of A1, even with extra explanation, count as correct.
2. If the question involves a geographical region, and A2 includes the information in A1, count as correct.
3. For numeric questions, treat "NA" as 0. If A1 is 0 and A2 is blank, count as correct.
4. If both A1 and A2 are blank, count as correct. Treat "", "NA", "Not Reported" or “None” as the same (correct).
5. If A1 contains "or", consider A2 correct if it matches any one of the options in A1 separated by "or".
. If A1 is "Not Reported", A2 can have assumed answer if evidence & rational both make sense, count as correct. 

Q: {q}

A1: {a1}

A2: {a2}
Evidence2: {evidence2}
Rationale2: {rationale2}
"""

        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful evaluator."},
                {"role": "user", "content": prompt}
            ]
        )

        # Assign directly to the DataFrame
        records.loc[idx, 'AI_eval'] = resp.choices[0].message.content.strip()
        if count == 60:
            break


    records.to_excel(f"Results_19/ai_auto_eval_19_{folder_name}.xlsx", index=False)

folder_name = "deepseek-chat-2"
auto_eval_answer(folder_name, model)